In [1]:
# 🎯 Example 1: Sequential Multi-Agent System (Pipeline)
# 🧠 Scenario

# “A travel company has different employees (agents):

# Planner → decides steps

# Flight Agent → finds flights

# Weather Agent → checks weather

# Decision Agent → gives final answer”

In [2]:
def planner_agent(user_query):
    print("\n[Planner Agent] Creating plan...")
    return ["flight", "weather", "decision"]

In [3]:
def flight_agent():
    print("\n[Flight Agent] Fetching flights...")
    return [
        {"airline": "IndiGo", "price": 4500},
        {"airline": "Air India", "price": 5200}
    ]


In [4]:
def weather_agent():
    print("\n[Weather Agent] Checking weather...")
    return {"condition": "Clear", "temp": 28}

In [5]:
def decision_agent(flights, weather):
    print("\n[Decision Agent] Making decision...")

    cheapest = min(flights, key=lambda x: x["price"])

    if weather["condition"] == "Rain":
        return "Avoid travel due to bad weather"

    return f"Book {cheapest['airline']} at ₹{cheapest['price']}"

In [6]:
def travel_multi_agent(user_query):
    print("User Query:", user_query)

    plan = planner_agent(user_query)

    flights = None
    weather = None

    for step in plan:
        if step == "flight":
            flights = flight_agent()

        elif step == "weather":
            weather = weather_agent()

        elif step == "decision":
            result = decision_agent(flights, weather)

    return result

In [7]:
response = travel_multi_agent("Plan my trip Delhi to Mumbai")
print("\nFinal Answer:", response)

User Query: Plan my trip Delhi to Mumbai

[Planner Agent] Creating plan...

[Flight Agent] Fetching flights...

[Weather Agent] Checking weather...

[Decision Agent] Making decision...

Final Answer: Book IndiGo at ₹4500


In [8]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)

# ================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions

# ================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines

# ================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient

In [9]:

def intake_agent(patient_query):
    print("\n[Intake Agent] Collecting patient details...")

    # Simulated extracted info
    patient_data = {
        "symptoms": patient_query,
        "history": "No major past illness"
    }

    # Plan steps
    plan = ["diagnosis", "treatment", "decision"]

    return patient_data, plan



def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Analyzing symptoms...")

    symptoms = patient_data["symptoms"].lower()

    if "fever" in symptoms and "cough" in symptoms:
        diagnosis = "Possible Flu"
    elif "chest pain" in symptoms:
        diagnosis = "Possible Heart Issue"
    else:
        diagnosis = "General Checkup Required"

    return {"diagnosis": diagnosis}




def treatment_agent(diagnosis_data):
    print("\n[Treatment Agent] Suggesting treatment...")

    diagnosis = diagnosis_data["diagnosis"]

    if diagnosis == "Possible Flu":
        treatment = "Rest + Paracetamol + Hydration"
    elif diagnosis == "Possible Heart Issue":
        treatment = "Immediate ECG + Doctor Consultation"
    else:
        treatment = "Routine tests and consultation"

    return {"treatment": treatment}




def decision_agent(patient_data, diagnosis_data, treatment_data):
    print("\n[Decision Agent] Final recommendation...")

    return f"""
Patient Symptoms: {patient_data['symptoms']}
Diagnosis: {diagnosis_data['diagnosis']}
Treatment Plan: {treatment_data['treatment']}
Final Advice: Follow the treatment and consult doctor if needed.
"""




def hospital_multi_agent(patient_query):
    print("Patient Query:", patient_query)

    # Step 1: Intake
    patient_data, plan = intake_agent(patient_query)

    diagnosis_data = None
    treatment_data = None
    final_result = None

    # Step 2: Execute plan sequentially
    for step in plan:
        if step == "diagnosis":
            diagnosis_data = diagnostic_agent(patient_data)

        elif step == "treatment":
            treatment_data = treatment_agent(diagnosis_data)

        elif step == "decision":
            final_result = decision_agent(
                patient_data, diagnosis_data, treatment_data
            )

    return final_result




response = hospital_multi_agent("Patient has fever and cough for 2 days")
print("\nFinal Answer:", response)

Patient Query: Patient has fever and cough for 2 days

[Intake Agent] Collecting patient details...

[Diagnostic Agent] Analyzing symptoms...

[Treatment Agent] Suggesting treatment...

[Decision Agent] Final recommendation...

Final Answer: 
Patient Symptoms: Patient has fever and cough for 2 days
Diagnosis: Possible Flu
Treatment Plan: Rest + Paracetamol + Hydration
Final Advice: Follow the treatment and consult doctor if needed.



In [10]:
# USING API

In [14]:
import requests
from dotenv import load_dotenv
import os

# ================================
# 🔐 LOAD API KEY
# ================================
load_dotenv()
API_TOKEN = os.getenv("HUGGINGFACE_API_KEY")
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

def query_llm(prompt: str):
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 200
    }

    response = requests.post(API_URL, headers=headers, json=payload)
    data = response.json()

    if "choices" in data:
        return data["choices"][0]["message"]["content"]
    else:
        return str(data)


# ================================
# 🧠 AGENT 1: INTAKE AGENT
# ================================
def intake_agent(patient_query):
    print("\n[Intake Agent] Collecting details...")

    thought = query_llm(f"""
    Extract key medical information from this patient query:
    {patient_query}
    """)

    plan = ["diagnosis", "treatment", "decision"]

    return {"raw": patient_query, "analysis": thought}, plan


# ================================
# 🔬 AGENT 2: DIAGNOSTIC AGENT
# ================================
def diagnostic_agent(patient_data):
    print("\n[Diagnostic Agent] Diagnosing...")

    diagnosis = query_llm(f"""
    Based on this patient information:
    {patient_data}

    What is the most likely condition?
    """)

    return {"diagnosis": diagnosis}


# ================================
# 💊 AGENT 3: TREATMENT AGENT
# ================================
def treatment_agent(diagnosis_data):
    print("\n[Treatment Agent] Suggesting treatment...")

    treatment = query_llm(f"""
    Given this diagnosis:
    {diagnosis_data}

    Suggest appropriate treatment options.
    """)

    return {"treatment": treatment}


# ================================
# 🧾 AGENT 4: DECISION AGENT
# ================================
def decision_agent(patient_data, diagnosis_data, treatment_data):
    print("\n[Decision Agent] Final decision...")

    final = query_llm(f"""
    Patient Info: {patient_data}
    Diagnosis: {diagnosis_data}
    Treatment: {treatment_data}

    Provide a final recommendation in a clear and concise way.
    """)

    return final


# ================================
# 🚀 MAIN PIPELINE
# ================================
def hospital_multi_agent(patient_query):
    print("Patient Query:", patient_query)

    # Step 1
    patient_data, plan = intake_agent(patient_query)

    diagnosis_data = None
    treatment_data = None
    final_result = None

    # Step 2: Sequential execution
    for step in plan:
        if step == "diagnosis":
            diagnosis_data = diagnostic_agent(patient_data)

        elif step == "treatment":
            treatment_data = treatment_agent(diagnosis_data)

        elif step == "decision":
            final_result = decision_agent(
                patient_data, diagnosis_data, treatment_data
            )

    return final_result


# ▶️ RUN
response = hospital_multi_agent("Patient has fever, cough and body pain for 3 days")
print("\nFinal Answer:\n", response)

Patient Query: Patient has fever, cough and body pain for 3 days

[Intake Agent] Collecting details...

[Diagnostic Agent] Diagnosing...

[Treatment Agent] Suggesting treatment...

[Decision Agent] Final decision...

Final Answer:
 **Final Recommendation:**

Based on the patient's symptoms and the analysis, the most likely diagnosis is an **Upper Respiratory Tract Infection (URTI)**. The recommended treatment plan is as follows:

1. **Rest and Hydration:** Encourage the patient to get plenty of rest and drink plenty of fluids, such as water, clear broths, or electrolyte-rich beverages, to stay hydrated.
2. **Over-the-Counter Medications:** Use acetaminophen (Tylenol) or ibuprofen (Advil, Motrin) to alleviate fever, headache, and body pain, following the recommended dosage and instructions.
3. **Decongestants and Expectorants:** Use decongestants, such as pseudoephedrine (Sudafed), to relieve nasal congestion, and expectorants, such as guaifenesin (Mucinex), as needed.

**No antibiotics

In [15]:
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market. The Manager Agent oversees the process and delegates tasks to specialized workers.

# 👩‍💼 Manager Agent
# - Receives the overall goal: “Evaluate feasibility of launching Product Y in Asia.”
# - Dynamically assigns tasks to worker agents depending on what’s needed.
# - Example: If budget is unclear → send to Finance Worker. If regulations are complex → send to Legal Worker.

# ================================
# 📊 Worker Agents
# ================================
# - Market Research Worker
# - Collects competitor data, customer preferences, and demand forecasts.
# - Reports: “High demand in Tier‑1 cities, moderate competition.”
# - Finance Worker
# - Analyzes budget, ROI, and pricing strategy.
# - Reports: “Estimated $10M investment, ROI in 2 years.”
# - Operations Worker
# - Evaluates supply chain, production capacity, and logistics.
# - Reports: “Factories can scale up, but shipping costs are high.”
# - Legal Worker
# - Reviews compliance, intellectual property, and regional regulations.
# - Reports: “Trademark available, but import laws require certification.”
# - HR Worker
# - Assesses staffing needs and training requirements.
# - Reports: “Need 50 new hires for customer support and sales.”

In [ ]:
# Without Api

In [18]:
# ================================
# Worker Agents
# ================================

class MarketResearchWorker:
    def analyze(self):
        return {
            "demand": "High demand in Tier-1 cities",
            "competition": "Moderate competition"
        }

class FinanceWorker:
    def analyze(self):
        return {
            "investment": "$10M required",
            "roi": "Expected ROI in 2 years"
        }

class OperationsWorker:
    def analyze(self):
        return {
            "production": "Factories can scale up",
            "logistics": "Shipping costs are high"
        }

class LegalWorker:
    def analyze(self):
        return {
            "trademark": "Available",
            "regulation": "Import laws require certification"
        }

class HRWorker:
    def analyze(self):
        return {
            "hiring": "Need 50 new hires",
            "roles": "Customer support and sales"
        }


# ================================
# Manager Agent
# ================================

class ManagerAgent:
    def __init__(self):
        self.market = MarketResearchWorker()
        self.finance = FinanceWorker()
        self.operations = OperationsWorker()
        self.legal = LegalWorker()
        self.hr = HRWorker()

    def evaluate_feasibility(self):
        print("Manager: Evaluating Product Y launch in Asia...\n")

        results = {}

        # Step 1: Market Analysis
        results["market"] = self.market.analyze()

        # Step 2: Finance Analysis
        results["finance"] = self.finance.analyze()

        # Step 3: Operations Analysis
        results["operations"] = self.operations.analyze()

        # Step 4: Legal Analysis
        results["legal"] = self.legal.analyze()

        # Step 5: HR Analysis
        results["hr"] = self.hr.analyze()

        return self.final_decision(results)

    def final_decision(self, results):
        print("📊 Consolidating Reports...\n")

        for key, value in results.items():
            print(f"{key.upper()} REPORT:", value)

        print("\n📌 Final Decision:")

        if "high" in results["operations"]["logistics"].lower():
            return "Launch possible, but reduce logistics cost before scaling."
        else:
            return "Launch immediately."


# ================================
# Run System
# ================================

manager = ManagerAgent()
decision = manager.evaluate_feasibility()

print("\n✅ Decision:", decision)

Manager: Evaluating Product Y launch in Asia...

📊 Consolidating Reports...

MARKET REPORT: {'demand': 'High demand in Tier-1 cities', 'competition': 'Moderate competition'}
FINANCE REPORT: {'investment': '$10M required', 'roi': 'Expected ROI in 2 years'}
OPERATIONS REPORT: {'production': 'Factories can scale up', 'logistics': 'Shipping costs are high'}
LEGAL REPORT: {'trademark': 'Available', 'regulation': 'Import laws require certification'}
HR REPORT: {'hiring': 'Need 50 new hires', 'roles': 'Customer support and sales'}

📌 Final Decision:

✅ Decision: Launch possible, but reduce logistics cost before scaling.


In [16]:
import requests
from dotenv import load_dotenv
import os

# ================================
# 🔐 LOAD API KEY
# ================================
load_dotenv()
API_TOKEN = os.getenv("HUGGINGFACE_API_KEY")
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

def query_llm(prompt: str):
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 200
    }

    response = requests.post(API_URL, headers=headers, json=payload)
    data = response.json()

    if "choices" in data:
        return data["choices"][0]["message"]["content"]
    else:
        return str(data)
 


# ================================
# 👩‍💼 MANAGER AGENT
# ================================
def manager_agent(goal):
    print("\n[Manager Agent] Analyzing goal...")

    plan = query_llm(f"""
    Goal: {goal}

    Decide which departments are needed from:
    Market Research, Finance, Operations, Legal, HR.

    Return as a comma-separated list.
    """)

    print("Plan:", plan)
    return plan.lower()


# ================================
# 📊 WORKER AGENTS
# ================================
def market_research_worker(goal):
    print("\n[Market Research Worker] Working...")
    return query_llm(f"""
    Analyze market demand, competition, and customer preferences for:
    {goal}
    """)


def finance_worker(goal):
    print("\n[Finance Worker] Working...")
    return query_llm(f"""
    Estimate investment, ROI, and pricing strategy for:
    {goal}
    """)


def operations_worker(goal):
    print("\n[Operations Worker] Working...")
    return query_llm(f"""
    Evaluate supply chain, production, and logistics for:
    {goal}
    """)


def legal_worker(goal):
    print("\n[Legal Worker] Working...")
    return query_llm(f"""
    Check regulations, compliance, and legal risks for:
    {goal}
    """)


def hr_worker(goal):
    print("\n[HR Worker] Working...")
    return query_llm(f"""
    Estimate hiring needs and training requirements for:
    {goal}
    """)


# ================================
# 🧾 DECISION AGENT
# ================================
def decision_agent(goal, reports):
    print("\n[Decision Agent] Final strategy...")

    return query_llm(f"""
    Goal: {goal}

    Reports:
    {reports}

    Give a final business recommendation with risks and feasibility.
    """)


# ================================
# 🚀 MAIN SYSTEM (DYNAMIC FLOW)
# ================================
def corporate_agent(goal):
    print("Goal:", goal)

    plan = manager_agent(goal)

    reports = {}

    # Dynamic delegation
    if "market" in plan:
        reports["market"] = market_research_worker(goal)

    if "finance" in plan:
        reports["finance"] = finance_worker(goal)

    if "operations" in plan:
        reports["operations"] = operations_worker(goal)

    if "legal" in plan:
        reports["legal"] = legal_worker(goal)

    if "hr" in plan:
        reports["hr"] = hr_worker(goal)

    # Final decision
    final = decision_agent(goal, reports)

    return final


# ▶️ RUN
result = corporate_agent("Evaluate feasibility of launching Product Y in Asia")
print("\nFinal Answer:\n", result)

Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Analyzing goal...
Plan: To evaluate the feasibility of launching Product Y in Asia, the following departments would be necessary:

Market Research, Finance, Operations, Legal 

These departments will be essential in assessing market demand and competition, providing financial projections, ensuring operational readiness, and navigating regulatory requirements, as well as compliance and employment laws in the region.

[Market Research Worker] Working...

[Finance Worker] Working...

[Operations Worker] Working...

[Legal Worker] Working...

[Decision Agent] Final strategy...

Final Answer:
 **Final Business Recommendation:**

Based on the reports from Market Analysis, Finance, Operations, and Legal departments, we recommend launching Product Y in Asia with caution and careful consideration of the following factors:

1. **Market Demand:** Asia's rapidly expanding middle class, urbanization, and increased disposable

In [19]:
# Agent 1: Crisis Coordinator (Broadcaster)
# - Broadcasts: “Data breach detected in customer database. Immediate response required.”
# - Sends this to all other agents simultaneously.

# 🛡️ Agent 2: IT Security Agent
# - Receives broadcast.
# - Responds: “Isolate affected servers, patch vulnerabilities, start forensic analysis.”

# 📞 Agent 3: Communications Agent
# - Receives broadcast.
# - Responds: “Draft internal memo, prepare press release, notify stakeholders.”

# 💰 Agent 4: Finance Agent
# - Receives broadcast.
# - Responds: “Estimate financial impact, allocate emergency funds, review insurance coverage.”

# 👩‍⚖️ Agent 5: Legal Agent
# - Receives broadcast.
# - Responds: “Assess regulatory obligations, prepare compliance reports, advise on liability.”

# 👩‍💼 Agent 6: HR Agent
# - Receives broadcast.
# - Responds: “Brief employees, provide guidance on handling customer queries, ensure morale support.”

# 🧑‍⚖️ Agent 7: Decision Agent (Coordinator)
# - Collects all responses.
# - Integrates into a final crisis response plan:
# “Servers isolated, communications prepared, financial impact assessed, compliance secured, employees briefed.”

In [2]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()  # loads .env file

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ================================
#  LLM FUNCTION
# ================================
def query_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return response.choices[0].message.content


# ================================
#  MANAGER AGENT
# ================================
def manager_agent(goal):
    print("\n[Manager Agent] Planning...")

    plan_text = query_llm(f"""
    You are a manager agent. Based on the goal below, decide which workers are needed.
    Available workers: market, finance, operations, legal, hr
    Return only a list like: market, finance, legal

    Goal: {goal}
    """)

    print("LLM Plan:", plan_text)

    # Convert LLM text → list
    plan = [p.strip().lower() for p in plan_text.split(",")]
    return plan


# ================================
#  MARKET WORKER
# ================================
def market_worker(goal):
    print("\n[Market Worker]")
    return query_llm(f"Analyze market demand and competition for: {goal}")


# ================================
#  FINANCE WORKER
# ================================
def finance_worker(goal):
    print("\n[Finance Worker]")
    return query_llm(f"Estimate budget, ROI, and pricing for: {goal}")


# ================================
#  OPERATIONS WORKER
# ================================
def operations_worker(goal):
    print("\n[Operations Worker]")
    return query_llm(f"Analyze production, supply chain, and logistics for: {goal}")


# ================================
#  LEGAL WORKER
# ================================
def legal_worker(goal):
    print("\n[Legal Worker]")
    return query_llm(f"Check regulations, compliance, and risks for: {goal}")


# ================================
#  HR WORKER
# ================================
def hr_worker(goal):
    print("\n[HR Worker]")
    return query_llm(f"Estimate hiring and staffing needs for: {goal}")


# ================================
#  DECISION AGENT
# ================================
def decision_agent(results):
    print("\n[Decision Agent] Final Strategy...")

    combined = "\n".join([f"{k}: {v}" for k, v in results.items()])

    return query_llm(f"""
    Based on the following reports, give final business decision:

    {combined}
    """)


# ================================
#  MAIN SYSTEM
# ================================
def corporate_agent(goal):
    print("Goal:", goal)

    plan = manager_agent(goal)

    results = {}

    for step in plan:

        if "market" in step:
            results["Market"] = market_worker(goal)

        elif "finance" in step:
            results["Finance"] = finance_worker(goal)

        elif "operations" in step:
            results["Operations"] = operations_worker(goal)

        elif "legal" in step:
            results["Legal"] = legal_worker(goal)

        elif "hr" in step:
            results["HR"] = hr_worker(goal)

    final_result = decision_agent(results)

    return final_result


# ================================
#  RUN
# ================================
response = corporate_agent("Evaluate feasibility of launching Product Y in Asia")
print("\nFinal Answer:\n", response)


Goal: Evaluate feasibility of launching Product Y in Asia

[Manager Agent] Planning...
LLM Plan: Based on the goal, I would need the following workers:

1. Market: To assess market demand and competition in Asia for Product Y.
2. Legal: To ensure compliance with local laws and regulations in Asia.
3. Finance: To evaluate the financial feasibility of launching Product Y in Asia, including costs, revenue projections, and potential return on investment.

These three workers would provide the necessary expertise to evaluate the feasibility of launching Product Y in Asia.

[Market Worker]

[HR Worker]

[Decision Agent] Final Strategy...

Final Answer:
 **Final Business Decision:**

Based on the market analysis and feasibility study, we recommend launching Product Y in Asia, with a focus on the following markets:

1. **China:** Given its large population, growing middle class, and increasing consumer spending power, China is an attractive market for Product Y.
2. **South Korea:** South Korea

In [5]:
def crisis_coordinator():
    message = "Data breach detected in customer database. Immediate response required."
    print("\n[Crisis Coordinator] Broadcasting alert...")
    return message


# ================================
#  IT SECURITY AGENT
# ================================
def it_security_agent(msg):
    print("\n[IT Security Agent] Responding...")
    return "Isolate servers, patch vulnerabilities, start forensic analysis"


# ================================
#  COMMUNICATIONS AGENT
# ================================
def communications_agent(msg):
    print("\n[Communications Agent] Responding...")
    return "Draft memo, prepare press release, notify stakeholders"


# ================================
#  FINANCE AGENT
# ================================
def finance_agent(msg):
    print("\n[Finance Agent] Responding...")
    return "Estimate financial impact, allocate emergency funds"


# ================================
#  LEGAL AGENT
# ================================
def legal_agent(msg):
    print("\n[Legal Agent] Responding...")
    return "Assess regulations, prepare compliance reports"


# ================================
#  HR AGENT
# ================================
def hr_agent(msg):
    print("\n[HR Agent] Responding...")
    return "Brief employees and provide support"


# ================================
#  DECISION AGENT
# ================================
def decision_agent(responses):
    print("\n[Decision Agent] Combining responses...")

    return f"""
Final Crisis Plan:
- IT: {responses['it']}
- Communications: {responses['comm']}
- Finance: {responses['finance']}
- Legal: {responses['legal']}
- HR: {responses['hr']}

Conclusion: Crisis handled with coordinated response.
"""


# ================================
#  MAIN SYSTEM (BROADCAST MODEL)
# ================================
def crisis_management_system():
    msg = crisis_coordinator()

    responses = {
        "it": it_security_agent(msg),
        "comm": communications_agent(msg),
        "finance": finance_agent(msg),
        "legal": legal_agent(msg),
        "hr": hr_agent(msg)
    }

    final = decision_agent(responses)
    return final


# RUN
print(crisis_management_system())




[Crisis Coordinator] Broadcasting alert...

[IT Security Agent] Responding...

[Communications Agent] Responding...

[Finance Agent] Responding...

[Legal Agent] Responding...

[HR Agent] Responding...

[Decision Agent] Combining responses...

Final Crisis Plan:
- IT: Isolate servers, patch vulnerabilities, start forensic analysis
- Communications: Draft memo, prepare press release, notify stakeholders
- Finance: Estimate financial impact, allocate emergency funds
- Legal: Assess regulations, prepare compliance reports
- HR: Brief employees and provide support

Conclusion: Crisis handled with coordinated response.



In [6]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()  # loads .env file

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


def query_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return response.choices[0].message.content


# ================================
#  CRISIS COORDINATOR
# ================================
def crisis_coordinator():
    msg = "Data breach detected in customer database. Immediate response required."
    print("\n[Crisis Coordinator] Broadcasting...")
    return msg


# ================================
#  IT SECURITY AGENT
# ================================
def it_security_agent(msg):
    print("\n[IT Security Agent]")
    return query_llm(f"What should IT team do in this situation: {msg}")


# ================================
#  COMMUNICATION AGENT
# ================================
def communications_agent(msg):
    print("\n[Communications Agent]")
    return query_llm(f"How should company communicate during: {msg}")


# ================================
#  FINANCE AGENT
# ================================
def finance_agent(msg):
    print("\n[Finance Agent]")
    return query_llm(f"What financial actions are needed for: {msg}")


# ================================
#  LEGAL AGENT
# ================================
def legal_agent(msg):
    print("\n[Legal Agent]")
    return query_llm(f"What legal steps are required for: {msg}")


# ================================
#  HR AGENT
# ================================
def hr_agent(msg):
    print("\n[HR Agent]")
    return query_llm(f"What HR actions are needed for: {msg}")


# ================================
#  DECISION AGENT
# ================================
def decision_agent(responses):
    print("\n[Decision Agent] Finalizing plan...")

    combined = "\n".join([f"{k}: {v}" for k, v in responses.items()])

    return query_llm(f"""
    Combine these responses into a final crisis response plan:

    {combined}
    """)


# ================================
#  MAIN SYSTEM
# ================================
def crisis_management_system():
    msg = crisis_coordinator()

    responses = {
        "IT": it_security_agent(msg),
        "Communications": communications_agent(msg),
        "Finance": finance_agent(msg),
        "Legal": legal_agent(msg),
        "HR": hr_agent(msg)
    }

    final = decision_agent(responses)
    return final


# RUN
print(crisis_management_system())



[Crisis Coordinator] Broadcasting...

[IT Security Agent]

[Communications Agent]

[Finance Agent]

[Legal Agent]

[HR Agent]

[Decision Agent] Finalizing plan...
**Crisis Response Plan for a Data Breach in a Customer Database**

**I. Initial Response (0-30 minutes)**

1. **Notify management and stakeholders**: Inform the management team, data protection officer, and relevant stakeholders about the breach.
2. **Activate incident response plan**: Follow the organization's incident response plan, which should include procedures for data breaches.
3. **Assess the breach**: Determine the scope, severity, and potential impact of the breach.
4. **Contain the breach**: Immediately restrict access to the affected system or database to prevent further unauthorized access.

**II. Data Collection and Analysis (30 minutes-2 hours)**

1. **Gather information**: Collect data on the breach, including the type of data compromised, the number of records affected, and the potential impact on customers.

In [7]:
# Scenario: Corporate Product Launch Broadcast
# Imagine a company preparing to launch Product X in Q3. The Coordinator Agent (like a corporate program manager) sends out a broadcast announcement to all departments at once:
# “Product X launch in Q3, target market North America, budget $5M.”


# 📢 Coordinator Agent (Broadcaster)
# - Sends the launch announcement to all departments simultaneously.
# - This is implemented in the code by the broadcast() function, which uses asyncio.gather() to run all agents in parallel.

# 📈 Marketing Agent
# - Receives the broadcast.
# - Responds with a marketing strategy: campaigns, channels, and positioning.
# 💰 Finance Agent
# - Receives the broadcast.
# - Responds with budget allocation and ROI forecasts.
# 🏭 Operations Agent
# - Receives the broadcast.
# - Responds with production and supply chain actions.
# 👩‍⚖️ Legal Agent
# - Receives the broadcast.
# - Responds with compliance checks and contract actions.
# 👩‍💼 HR Agent
# - Receives the broadcast.
# - Responds with staffing and training plans.

# 🧑‍⚖️ Decision Agent
# - Collects all responses.
# - Integrates them into a Final Corporate Launch Plan.
# - In the code, this is the decision_agent() function that merges all outputs into one consolidated plan.



# ================================
#  COORDINATOR AGENT (BROADCAST)
# ================================
def coordinator_agent():
    msg = "Product X launch in Q3, target market North America, budget $5M"
    print("\n[Coordinator] Broadcasting launch plan...")
    return msg


# ================================
#  MARKETING AGENT
# ================================
def marketing_agent(msg):
    print("[Marketing Agent] Processing...")
    return "Run digital campaigns, social media ads, premium positioning"


# ================================
#  FINANCE AGENT
# ================================
def finance_agent(msg):
    print("[Finance Agent] Processing...")
    return "Allocate $5M budget, expected ROI in 18 months"


# ================================
#  OPERATIONS AGENT
# ================================
def operations_agent(msg):
    print("[Operations Agent] Processing...")
    return "Scale production and optimize supply chain"


# ================================
#  LEGAL AGENT
# ================================
def legal_agent(msg):
    print("[Legal Agent] Processing...")
    return "Ensure compliance and finalize contracts"


# ================================
#  HR AGENT
# ================================
def hr_agent(msg):
    print("[HR Agent] Processing...")
    return "Hire and train sales & support teams"


# ================================
#  DECISION AGENT
# ================================
def decision_agent(responses):
    print("\n[Decision Agent] Combining all responses...")

    return f"""
Final Corporate Launch Plan:
- Marketing: {responses['marketing']}
- Finance: {responses['finance']}
- Operations: {responses['operations']}
- Legal: {responses['legal']}
- HR: {responses['hr']}

Conclusion: Product launch strategy ready for Q3.
"""


# ================================
#  MAIN SYSTEM (SEQUENTIAL FLOW)
# ================================
def corporate_broadcast_system():
    msg = coordinator_agent()

    responses = {}

    # Sequential execution (no asyncio)
    responses["marketing"] = marketing_agent(msg)
    responses["finance"] = finance_agent(msg)
    responses["operations"] = operations_agent(msg)
    responses["legal"] = legal_agent(msg)
    responses["hr"] = hr_agent(msg)

    final = decision_agent(responses)
    return final


# RUN
result = corporate_broadcast_system()
print(result)


[Coordinator] Broadcasting launch plan...
[Marketing Agent] Processing...
[Finance Agent] Processing...
[Operations Agent] Processing...
[Legal Agent] Processing...
[HR Agent] Processing...

[Decision Agent] Combining all responses...

Final Corporate Launch Plan:
- Marketing: Run digital campaigns, social media ads, premium positioning
- Finance: Allocate $5M budget, expected ROI in 18 months
- Operations: Scale production and optimize supply chain
- Legal: Ensure compliance and finalize contracts
- HR: Hire and train sales & support teams

Conclusion: Product launch strategy ready for Q3.



In [8]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()  # loads .env file

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


# ================================
#  LLM FUNCTION
# ================================
def query_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5
    )
    return response.choices[0].message.content


# ================================
#  COORDINATOR AGENT (BROADCAST)
# ================================
def coordinator_agent():
    msg = "Product X launch in Q3, target market North America, budget $5M"
    print("\n[Coordinator] Broadcasting launch plan...")
    return msg


# ================================
#  MARKETING AGENT
# ================================
def marketing_agent(msg):
    print("[Marketing Agent] Processing...")
    return query_llm(f"Create a marketing strategy for: {msg}")


# ================================
#  FINANCE AGENT
# ================================
def finance_agent(msg):
    print("[Finance Agent] Processing...")
    return query_llm(f"Plan budget allocation and ROI for: {msg}")


# ================================
#  OPERATIONS AGENT
# ================================
def operations_agent(msg):
    print("[Operations Agent] Processing...")
    return query_llm(f"Plan production and supply chain for: {msg}")


# ================================
#  LEGAL AGENT
# ================================
def legal_agent(msg):
    print("[Legal Agent] Processing...")
    return query_llm(f"Check compliance and legal requirements for: {msg}")


# ================================
#  HR AGENT
# ================================
def hr_agent(msg):
    print("[HR Agent] Processing...")
    return query_llm(f"Plan hiring and training for: {msg}")


# ================================
#  DECISION AGENT
# ================================
def decision_agent(responses):
    print("\n[Decision Agent] Combining all responses...")

    combined = "\n".join([f"{k}: {v}" for k, v in responses.items()])

    return query_llm(f"""
    Combine the following into a final corporate product launch plan:

    {combined}
    """)


# ================================
#  MAIN SYSTEM (SEQUENTIAL)
# ================================
def corporate_broadcast_system():
    msg = coordinator_agent()

    responses = {}

    # Sequential execution
    responses["Marketing"] = marketing_agent(msg)
    responses["Finance"] = finance_agent(msg)
    responses["Operations"] = operations_agent(msg)
    responses["Legal"] = legal_agent(msg)
    responses["HR"] = hr_agent(msg)

    final = decision_agent(responses)
    return final


# ================================
#  RUN
# ================================
result = corporate_broadcast_system()
print("\nFinal Answer:\n", result)


[Coordinator] Broadcasting launch plan...
[Marketing Agent] Processing...
[Finance Agent] Processing...
[Operations Agent] Processing...
[Legal Agent] Processing...
[HR Agent] Processing...

[Decision Agent] Combining all responses...

Final Answer:
 Here is the combined corporate product launch plan:

**Product X Launch Plan**

**Executive Summary:**

Product X is a revolutionary new product that is poised to disrupt the market with its innovative features and benefits. Our marketing strategy aims to create buzz and excitement among the target audience in North America, drive sales, and establish Product X as a leader in its category. With a budget of $5M, we will execute a multi-channel marketing plan that leverages digital, social, and traditional media to reach our target audience.

**Target Market:**

* Demographics: 18-45 years old, middle to upper-income households
* Psychographics: Tech-savvy, innovative, and open to new experiences
* Geographic location: North America (USA an